In [2]:
!pip install pandas numpy scikit-learn matplotlib seaborn


In [3]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)
n_samples = 1000

# Generate chaotic raw data
data = {
    'Customer_ID': range(1001, 1001 + n_samples),
    'Age': np.random.choice([np.nan, 25, 34, 45, 52, 120, -5], size=n_samples, p=[0.1, 0.2, 0.3, 0.2, 0.15, 0.02, 0.03]),
    'Income': np.random.normal(loc=55000, scale=15000, size=n_samples),
    'Join_Date': pd.date_range(start='2023-01-01', periods=n_samples, freq='h').strftime('%Y-%m-%d %H:%M:%S'),
    'Total_Spend': np.random.exponential(scale=200, size=n_samples),
    'Purchase_Count': np.random.randint(1, 50, size=n_samples).astype(float),
    'City': np.random.choice(['New York', 'Los Angeles', 'Chicago', None], size=n_samples, p=[0.4, 0.3, 0.2, 0.1])
}

# Inject specific anomalies
df = pd.DataFrame(data)
df.loc[df['Income'] < 20000, 'Income'] = np.nan  # Inject missing values in Income
df.loc[np.random.choice(df.index, 30), 'Income'] = 999999  # Inject extreme outliers in Income
df.loc[np.random.choice(df.index, 50), 'Purchase_Count'] = np.nan  # Inject missing values in Purchase Count

# Save to CSV
df.to_csv('raw_data.csv', index=False)
print("Chaotic dataset 'raw_data.csv' successfully created!")


Chaotic dataset 'raw_data.csv' successfully created!


In [4]:
# Load the dataset
df = pd.read_csv('raw_data.csv')

# View the first 5 rows
df.head()


,Customer_ID,Age,Income,Join_Date,Total_Spend,Purchase_Count,City
0,1001,34.0,57665.515014,2023-01-01 00:00:00,711.799780,30.0,New York
1,1002,120.0,34969.834619,2023-01-01 01:00:00,80.498010,40.0,New York
2,1003,45.0,60702.967765,2023-01-01 02:00:00,131.571861,18.0,Los Angeles
3,1004,34.0,64158.786179,2023-01-01 03:00:00,43.655510,48.0,New York
4,1005,25.0,63396.856719,2023-01-01 04:00:00,188.722145,2.0,New York


In [5]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Confirming your loaded dataframe is named 'df'
print("🔄 Starting pipeline processing...")

# ==========================================
# STEP 3: STATISTICAL MISSING DATA IMPUTATION
# ==========================================
# 1. Median Imputation for 'Age'
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. KNN Imputation for complex numerical fields ('Income', 'Purchase_Count')
imputer = KNNImputer(n_neighbors=5)
num_cols = ['Income', 'Purchase_Count', 'Total_Spend']
df[num_cols] = imputer.fit_transform(df[num_cols])

# 3. Mode Imputation for the categorical 'City' column
df['City'] = df['City'].fillna(df['City'].mode()[0])


# ==========================================
# STEP 4: OUTLIER DETECTION & NEUTRALIZATION
# ==========================================
# 1. Fix structural boundary errors in 'Age' (-5 and 120)
df.loc[df['Age'] < 0, 'Age'] = df['Age'].median()
df.loc[df['Age'] > 100, 'Age'] = df['Age'].median()

# 2. IQR Capping for 'Income' to neutralize extreme outliers
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df['Income'] = np.clip(df['Income'], lower_bound, upper_bound)


# ==========================================
# STEP 5: ADVANCED FEATURE ENGINEERING
# ==========================================
# Feature 1: Average Spend per Purchase (Mathematical combination)
df['Avg_Spend_Per_Purchase'] = df['Total_Spend'] / df['Purchase_Count']

# Feature 2: Income-to-Spend Ratio (Financial indicator)
df['Income_Spend_Ratio'] = df['Total_Spend'] / (df['Income'] + 1)

# Feature 3: Age Group Demographics (Continuous data binning)
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 30, 50, 100], labels=[1, 2, 3]).astype(int)


# ==========================================
# STEP 6: CLEAN EXPORT & VERIFICATION
# ==========================================
# Save output file
df.to_csv('cleaned_data.csv', index=False)

print("\n🚀 PIPELINE RUN COMPLETE!")
print(f"Data Dimensions: {df.shape}")
print(f"Remaining Missing Values: {df.isnull().sum().sum()}")
print("\nGenerated Features Preview:")
print(df[['Avg_Spend_Per_Purchase', 'Income_Spend_Ratio', 'Age_Group']].head())
print("\n💾 Cleaned dataset saved as 'cleaned_data.csv'!")


🔄 Starting pipeline processing...

🚀 PIPELINE RUN COMPLETE!
Data Dimensions: (1000, 10)
Remaining Missing Values: 0

Generated Features Preview:
   Avg_Spend_Per_Purchase  Income_Spend_Ratio  Age_Group
0               23.726659            0.012343          2
1                2.012450            0.002302          2
2                7.309548            0.002167          2
3                0.909490            0.000680          2
4               94.361073            0.002977          1

💾 Cleaned dataset saved as 'cleaned_data.csv'!
